# Customer Review Sentiment Analysis

**Objective:** Preprocess text, compare Naive Bayes and Logistic Regression using TF-IDF, evaluate and inspect errors.

> Dataset note: included CSVs are synthetic practice data created for this portfolio. Replace with approved public datasets for official internship submission.


## Setup
Run the following cell. Install missing packages with `pip install -r ../../requirements.txt` from the repository root.

In [ ]:
import pandas as pd, re, matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report,confusion_matrix,accuracy_score
df=pd.read_csv('customer_reviews_synthetic.csv')
display(df.Sentiment.value_counts())
df.Sentiment.value_counts().plot(kind='bar',title='Sentiment distribution'); plt.show()
def clean(t): return re.sub(r'[^a-z ]',' ',str(t).lower()).strip()
X_train,X_test,y_train,y_test=train_test_split(df.Text.map(clean),df.Sentiment,test_size=.2,random_state=42,stratify=df.Sentiment)
# TF-IDF converts text into weighted numeric word features; frequent words across all reviews receive less weight.
vec=TfidfVectorizer(stop_words='english',ngram_range=(1,2),max_features=5000)
A=vec.fit_transform(X_train); B=vec.transform(X_test)
models={'Naive Bayes':MultinomialNB(),'Logistic Regression':LogisticRegression(max_iter=1000)}
for name,m in models.items():
    m.fit(A,y_train); pred=m.predict(B)
    print(name,'accuracy',accuracy_score(y_test,pred)); print(classification_report(y_test,pred))
    sns.heatmap(confusion_matrix(y_test,pred,labels=['negative','neutral','positive']),annot=True,fmt='d',
                xticklabels=['negative','neutral','positive'],yticklabels=['negative','neutral','positive'])
    plt.title(name+' confusion matrix'); plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.show()
best=models['Logistic Regression']; errors=pd.DataFrame({'text':X_test,'actual':y_test,'predicted':best.predict(B)})
display(errors[errors.actual!=errors.predicted].head(5))
print('Error analysis: short, mixed-polarity and context-dependent statements are often ambiguous. Best model should be selected using F1 as well as accuracy.')
